In [1]:
import os
import re
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# Configuration
# ============================================================

BASE_DIR = "/scratch/elena/9Li/results"
FV_TAG = "FV_1"

RUNS_BY_BEAM = {
    "-270 MeV/c": [2407, 2408, 2409, 2432],
    "-350 MeV/c": [2374, 2379],
}

# Cylindrical selection used for the vertex plots.
R_CYLINDER_CM = 153.7963
H_CYLINDER_CM = 271.4235

# Final FV box used by merge_and_fv_cut.py, in cm.
FV_BOX = {
    "x": (-20.0, 20.0),
    "y": (-20.0, 20.0),
    "z": (-130.0, 0.0),
}

RUNS = [
    (beam, run)
    for beam, runs in RUNS_BY_BEAM.items()
    for run in runs
]

print(f"Runs to inspect: {len(RUNS)}")
for beam, run in RUNS:
    print(f"  {beam:12s} | Run {run}")

Runs to inspect: 6
  -270 MeV/c   | Run 2407
  -270 MeV/c   | Run 2408
  -270 MeV/c   | Run 2409
  -270 MeV/c   | Run 2432
  -350 MeV/c   | Run 2374
  -350 MeV/c   | Run 2379


In [2]:
# ============================================================
# Exact file matching
# ============================================================

def get_stage_files(processed_dir, stage, sample):
    """
    Return only the intended files for one stage and sample.
    This avoids counting derived files as Stage 1 candidates.
    """
    is_bkg = sample == "bkg"

    if stage == "stage1":
        pattern = os.path.join(processed_dir, "Li9_clusters_chunk_*.pkl")
        regex = (
            r"Li9_clusters_chunk_\d+_BKG\.pkl"
            if is_bkg
            else r"Li9_clusters_chunk_\d+\.pkl"
        )

    elif stage == "multilat":
        pattern = os.path.join(processed_dir, "Li9_clusters_chunk_*_multilat.pkl")
        regex = (
            r"Li9_clusters_chunk_\d+_BKG_multilat\.pkl"
            if is_bkg
            else r"Li9_clusters_chunk_\d+_multilat\.pkl"
        )

    elif stage == "refine":
        pattern = os.path.join(processed_dir, "Refine_Li9_clusters_chunk_*.pkl")
        regex = (
            r"Refine_Li9_clusters_chunk_\d+_BKG\.pkl"
            if is_bkg
            else r"Refine_Li9_clusters_chunk_\d+\.pkl"
        )

    else:
        raise ValueError(f"Unknown stage: {stage}")

    return sorted(
        f for f in glob(pattern)
        if re.fullmatch(regex, os.path.basename(f))
    )


def read_pickle_files(files):
    """Read and concatenate pickle files, preserving an empty result if needed."""
    frames = []

    for path in files:
        try:
            df = pd.read_pickle(path)
            if not isinstance(df, pd.DataFrame):
                print(f"WARNING: not a DataFrame: {path}")
                continue
            frames.append(df)
        except Exception as exc:
            print(f"ERROR reading {path}: {exc}")

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


# ============================================================
# Load all stages
# ============================================================

data = {}
inventory_rows = []

for beam, run in RUNS:
    processed_dir = os.path.join(BASE_DIR, f"run{run}", "processed")
    fv_dir = os.path.join(processed_dir, FV_TAG)

    for sample in ("signal", "bkg"):
        stage_files = {
            stage: get_stage_files(processed_dir, stage, sample)
            for stage in ("stage1", "multilat", "refine")
        }

        final_name = (
            f"Final_FV_Li9_clusters_run{run}_BKG.pkl"
            if sample == "bkg"
            else f"Final_FV_Li9_clusters_run{run}.pkl"
        )
        final_path = os.path.join(fv_dir, final_name)

        dfs = {
            stage: read_pickle_files(files)
            for stage, files in stage_files.items()
        }
        dfs["final"] = (
            pd.read_pickle(final_path)
            if os.path.isfile(final_path)
            else pd.DataFrame()
        )

        data[(run, sample)] = {
            "beam": beam,
            "processed_dir": processed_dir,
            "fv_dir": fv_dir,
            "files": stage_files,
            "final_path": final_path,
            "dfs": dfs,
        }

        for stage in ("stage1", "multilat", "refine", "final"):
            df = dfs[stage]
            n_files = (
                len(stage_files[stage])
                if stage in stage_files
                else int(os.path.isfile(final_path))
            )

            inventory_rows.append({
                "beam": beam,
                "run": run,
                "sample": sample,
                "stage": stage,
                "files": n_files,
                "rows": len(df),
            })

inventory = pd.DataFrame(inventory_rows)

print("FILE AND ROW INVENTORY")
display(
    inventory.pivot_table(
        index=["beam", "run", "sample"],
        columns="stage",
        values=["files", "rows"],
        aggfunc="first",
        fill_value=0,
    )
)

FILE AND ROW INVENTORY


files                         rows                   \
stage                  final multilat refine stage1 final multilat  refine   
beam       run  sample                                                       
-270 MeV/c 2407 bkg        1       23     23     23   415   622711  622711   
                signal     1        5      5      5    95   122999  122999   
           2408 bkg        1       28     28     28   305   471733  471733   
                signal     1        6      6      6    61    95924   95924   
           2409 bkg        1       33     33     33   120   268614  268614   
                signal     1        8      8      8    33    65169   65169   
           2432 bkg        1       13     13     13   244   309198  309198   
                signal     1        3      3      3    49    73039   73039   
-350 MeV/c 2374 bkg        1       14     14     14   406   758895  758895   
                signal     1        7      7      7   182   358447  358447   
           2379 bkg        1       18     18     18    52   141078  141078   
                signal     1       26     26     26    98   202333  202333   

                                
stage                   stage1  
beam       run  sample          
-270 MeV/c 2407 bkg     622711  
                signal  122999  
           2408 bkg     471733  
                signal   95924  
           2409 bkg     268614  
                signal   65169  
           2432 bkg     309198  
                signal   73039  
-350 MeV/c 2374 bkg     758895  
                signal  358447  
           2379 bkg     141078  
                signal  202333

In [4]:
# ============================================================
# Row retention across stages
# ============================================================

stages = ["stage1", "multilat", "refine", "final"]
retention_rows = []

for (run, sample), info in data.items():
    dfs = info["dfs"]
    counts = {stage: len(dfs[stage]) for stage in stages}

    n_stage1 = counts["stage1"]

    retention_rows.append({
        "beam": info["beam"],
        "run": run,
        "sample": sample,
        "stage1": counts["stage1"],
        "multilat": counts["multilat"],
        "refine": counts["refine"],
        "final": counts["final"],
        "multilat/stage1 (%)": (
            100 * counts["multilat"] / n_stage1 if n_stage1 else 0
        ),
        "refine/multilat (%)": (
            100 * counts["refine"] / counts["multilat"]
            if counts["multilat"] else 0
        ),
        "final/refine (%)": (
            100 * counts["final"] / counts["refine"]
            if counts["refine"] else 0
        ),
        "final/stage1 (%)": (
            100 * counts["final"] / n_stage1 if n_stage1 else 0
        ),
    })

retention = pd.DataFrame(retention_rows)

display(
    retention.sort_values(["beam", "run", "sample"])
    .round(4)
)

,beam,run,sample,stage1,multilat,refine,final,multilat/stage1 (%),refine/multilat (%),final/refine (%),final/stage1 (%)
1,-270 MeV/c,2407,bkg,622711,622711,622711,415,100.0,100.0,0.0666,0.0666
0,-270 MeV/c,2407,signal,122999,122999,122999,95,100.0,100.0,0.0772,0.0772
3,-270 MeV/c,2408,bkg,471733,471733,471733,305,100.0,100.0,0.0647,0.0647
2,-270 MeV/c,2408,signal,95924,95924,95924,61,100.0,100.0,0.0636,0.0636
5,-270 MeV/c,2409,bkg,268614,268614,268614,120,100.0,100.0,0.0447,0.0447
4,-270 MeV/c,2409,signal,65169,65169,65169,33,100.0,100.0,0.0506,0.0506
7,-270 MeV/c,2432,bkg,309198,309198,309198,244,100.0,100.0,0.0789,0.0789
6,-270 MeV/c,2432,signal,73039,73039,73039,49,100.0,100.0,0.0671,0.0671
9,-350 MeV/c,2374,bkg,758895,758895,758895,406,100.0,100.0,0.0535,0.0535
8,-350 MeV/c,2374,signal,358447,358447,358447,182,100.0,100.0,0.0508,0.0508


In [6]:
# ============================================================
# Inspect final candidates and their fit quality
# ============================================================

for (run, sample), info in sorted(data.items()):
    df = info["dfs"]["final"]

    print(
        f"\nRun {run} | {sample} | "
        f"beam={info['beam']} | rows={len(df)}"
    )

    if df.empty:
        print("  Empty final DataFrame")
        continue

    print("  Columns:")
    print(df.columns.tolist())

    quality_cols = [
        c for c in [
            "fit_success",
            "chi2_ndof",
            "t_rms_fine",
            "nHits",
            "hits_after",
            "v_x_fine",
            "v_y_fine",
            "v_z_fine",
        ]
        if c in df.columns
    ]

    if quality_cols:
        print(df[quality_cols].describe(include="all").to_string())
    else:
        print("  No recognised fit-quality columns found.")


Run 2374 | bkg | beam=-350 MeV/c | rows=406
  Columns:
['t_window_start_ns', 't_window_start_rel_ns', 'nHits', 't_rms_ns', 'spill_id', 'nCLusters_in_spill', 'hit_slot_ids', 'hit_position_ids', 'hit_times_ns', 'hit_charges', 'vertex_x', 'vertex_y', 'vertex_z', 'fit_success', 'n_hits_used', 'time_rms', 'chi2', 'ndof', 'chi2_ndof', 'v_x_fine', 'v_y_fine', 'v_z_fine', 't_rms_fine', 'hits_after', 'nhits_clean', 'refinement_success', 'refinement_status', 'chi2_fine', 'ndof_fine', 'chi2_ndof_fine', 'p_value_fine', 'initial_rms_pass_3ns']
       fit_success   chi2_ndof  t_rms_fine       nHits  hits_after    v_x_fine    v_y_fine    v_z_fine
count          406  406.000000  406.000000  406.000000  406.000000  406.000000  406.000000  406.000000
unique           1         NaN         NaN         NaN         NaN         NaN         NaN         NaN
top           True         NaN         NaN         NaN         NaN         NaN         NaN         NaN
freq           406         NaN         NaN        

In [7]:
# ============================================================
# Fit and refinement quality audit across all stages
# ============================================================

import numpy as np
import pandas as pd

quality_rows = []

for (run, sample), info in sorted(data.items()):

    for stage in ("multilat", "refine", "final"):

        df = info["dfs"][stage]

        row = {
            "beam": info["beam"],
            "run": run,
            "sample": sample,
            "stage": stage,
            "rows": len(df),
        }

        if df.empty:
            quality_rows.append(row)
            continue

        # ----------------------------------------------------
        # Initial multilaterator fit
        # ----------------------------------------------------
        if "fit_success" in df.columns:
            row["fit_success_true"] = int(
                df["fit_success"].fillna(False).astype(bool).sum()
            )

            row["fit_success_false"] = int(
                (~df["fit_success"].fillna(False).astype(bool)).sum()
            )

        if "chi2_ndof" in df.columns:

            chi2 = pd.to_numeric(
                df["chi2_ndof"], errors="coerce"
            )

            row["chi2_ndof_finite"] = int(
                np.isfinite(chi2).sum()
            )

            row["chi2_ndof_lt5"] = int(
                (np.isfinite(chi2) & (chi2 < 5)).sum()
            )

        # ----------------------------------------------------
        # Refinement
        # ----------------------------------------------------
        if "refinement_success" in df.columns:

            row["refinement_success_true"] = int(
                df["refinement_success"]
                .fillna(False)
                .astype(bool)
                .sum()
            )

            row["refinement_success_false"] = int(
                (
                    ~df["refinement_success"]
                    .fillna(False)
                    .astype(bool)
                ).sum()
            )

        if "chi2_ndof_fine" in df.columns:

            chi2_fine = pd.to_numeric(
                df["chi2_ndof_fine"], errors="coerce"
            )

            row["chi2_ndof_fine_finite"] = int(
                np.isfinite(chi2_fine).sum()
            )

            row["chi2_ndof_fine_lt5"] = int(
                (
                    np.isfinite(chi2_fine)
                    & (chi2_fine < 5)
                ).sum()
            )

        # ----------------------------------------------------
        # Initial RMS cut: recorded, not applied
        # ----------------------------------------------------
        if "initial_rms_pass_3ns" in df.columns:

            rms_pass = df["initial_rms_pass_3ns"]

            row["initial_rms_pass_3ns_true"] = int(
                rms_pass.fillna(False).astype(bool).sum()
            )

            row["initial_rms_pass_3ns_false"] = int(
                (
                    ~rms_pass.fillna(False).astype(bool)
                ).sum()
            )

        quality_rows.append(row)


quality_audit = pd.DataFrame(quality_rows)

display(
    quality_audit.sort_values(
        ["beam", "run", "sample", "stage"]
    ).reset_index(drop=True)
)

/tmp/ipykernel_1532760/3297700045.py:61: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_1532760/3297700045.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_1532760/3297700045.py:99: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rms_pass.fillna(False).astype(bool).sum()
/tmp/ip

,beam,run,sample,stage,rows,fit_success_true,fit_success_false,chi2_ndof_finite,chi2_ndof_lt5,refinement_success_true,refinement_success_false,chi2_ndof_fine_finite,chi2_ndof_fine_lt5,initial_rms_pass_3ns_true,initial_rms_pass_3ns_false
0,-270 MeV/c,2407,bkg,final,415,415,0,415,330,415.0,0.0,415.0,415.0,388.0,27.0
1,-270 MeV/c,2407,bkg,multilat,622711,301876,320835,301876,109400,NaN,NaN,NaN,NaN,NaN,NaN
2,-270 MeV/c,2407,bkg,refine,622711,301876,320835,301876,109400,22139.0,600572.0,22139.0,22139.0,170548.0,452163.0
3,-270 MeV/c,2407,signal,final,95,95,0,95,76,95.0,0.0,95.0,95.0,86.0,9.0
4,-270 MeV/c,2407,signal,multilat,122999,59634,63365,59634,21477,NaN,NaN,NaN,NaN,NaN,NaN
5,-270 MeV/c,2407,signal,refine,122999,59634,63365,59634,21477,4298.0,118701.0,4298.0,4297.0,33440.0,89559.0
6,-270 MeV/c,2408,bkg,final,305,305,0,305,252,305.0,0.0,305.0,305.0,286.0,19.0
7,-270 MeV/c,2408,bkg,multilat,471733,231782,239951,231782,96921,NaN,NaN,NaN,NaN,NaN,NaN
8,-270 MeV/c,2408,bkg,refine,471733,231782,239951,231782,96921,20158.0,451575.0,20158.0,20156.0,142243.0,329490.0
9,-270 MeV/c,2408,signal,final,61,61,0,61,51,61.0,0.0,61.0,61.0,59.0,2.0


In [8]:
# ============================================================
# Compare candidate identities across stages
# ============================================================

KEY_COLS = ["spill_id", "t_window_start_ns"]

identity_rows = []

for (run, sample), info in sorted(data.items()):

    dfs = info["dfs"]

    available_stages = [
        stage
        for stage in ("stage1", "multilat", "refine", "final")
        if not dfs[stage].empty
    ]

    for stage in available_stages:

        df = dfs[stage]

        if not all(col in df.columns for col in KEY_COLS):
            identity_rows.append({
                "run": run,
                "sample": sample,
                "stage": stage,
                "rows": len(df),
                "unique_keys": np.nan,
                "duplicate_key_rows": np.nan,
            })
            continue

        keys = pd.MultiIndex.from_frame(
            df[KEY_COLS]
        )

        identity_rows.append({
            "run": run,
            "sample": sample,
            "stage": stage,
            "rows": len(df),
            "unique_keys": keys.nunique(),
            "duplicate_key_rows": int(keys.duplicated().sum()),
        })

identity_audit = pd.DataFrame(identity_rows)

display(
    identity_audit.sort_values(
        ["run", "sample", "stage"]
    ).reset_index(drop=True)
)

,run,sample,stage,rows,unique_keys,duplicate_key_rows
0,2374,bkg,final,406,406,0
1,2374,bkg,multilat,758895,758895,0
2,2374,bkg,refine,758895,758895,0
3,2374,bkg,stage1,758895,758895,0
4,2374,signal,final,182,182,0
5,2374,signal,multilat,358447,358447,0
6,2374,signal,refine,358447,358447,0
7,2374,signal,stage1,358447,358447,0
8,2379,bkg,final,52,52,0
9,2379,bkg,multilat,141078,141078,0
